In [ ]:
# task5_tools.py - 투자 판단 분석 도구
from utils import *
from datetime import datetime, timedelta
import pandas as pd
import yfinance as yf
import requests
from typing import Dict, Any, List, Tuple, Optional
import concurrent.futures
import numpy as np
import time
import random
import json

# Task5 Function Definitions for Function Calling
task5_functions = [
    {
        "name": "extract_stock_info",
        "description": "사용자 질문에서 종목명과 분석 의도를 추출하고 종목코드로 매핑합니다. KR/US 시장 모두 지원하며, 상장 여부를 확인합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "user_query": {
                    "type": "string",
                    "description": "사용자가 입력한 원본 질문"
                },
                "stock_name": {
                    "type": "string",
                    "description": "추출된 종목명 (예: '삼성전자', 'Apple', 'AAPL')"
                },
                "analysis_intent": {
                    "type": "string",
                    "enum": ["buy_decision", "past_buy_evaluation", "general_analysis", "risk_assessment", "valuation"],
                    "description": "분석 목적: buy_decision(매수 판단), past_buy_evaluation(과거 매수 평가), general_analysis(일반 분석), risk_assessment(리스크 평가), valuation(가치평가)"
                },
                "purchase_price": {
                    "type": "number",
                    "description": "매수 가격 (과거 매수 평가시 필요, 선택사항)"
                },
                "purchase_date": {
                    "type": "string", 
                    "format": "date",
                    "description": "매수 일자 (YYYY-MM-DD 형식, 과거 매수 평가시 필요, 선택사항)"
                }
            },
            "required": ["user_query", "stock_name", "analysis_intent"]
        }
    },
    {
        "name": "get_yfinance_data",
        "description": "Yahoo Finance API를 통해 기업의 성장성 지표를 수집합니다. 시세, 재무제표, 애널리스트 의견, 뉴스 등을 포함합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "Yahoo Finance 티커 코드 (예: '005930.KS', 'AAPL')"
                },
                "data_types": {
                    "type": "array",
                    "items": {
                        "type": "string",
                        "enum": ["price_data", "financial_statements", "analyst_info", "news", "technical_indicators"]
                    },
                    "default": ["price_data", "financial_statements", "analyst_info", "news"],
                    "description": "수집할 데이터 유형 선택"
                },
                "period": {
                    "type": "string",
                    "enum": ["1mo", "3mo", "6mo", "1y", "2y"],
                    "default": "1y",
                    "description": "데이터 수집 기간"
                }
            },
            "required": ["ticker"]
        }
    },
    {
        "name": "get_dart_disclosure_data",
        "description": "DART 전자공시시스템에서 기업의 실적 및 리스크 지표를 수집합니다. 재무제표, 감사보고서, 공시 정보 등을 포함합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "corp_code": {
                    "type": "string",
                    "description": "DART 기업 코드 (8자리)"
                },
                "corp_name": {
                    "type": "string",
                    "description": "기업명 (기업 코드가 없을 경우 사용)"
                },
                "data_types": {
                    "type": "array",
                    "items": {
                        "type": "string",
                        "enum": ["company_info", "financial_summary", "multi_year_financial", "audit_report", "major_disclosure"]
                    },
                    "default": ["company_info", "financial_summary", "audit_report"],
                    "description": "수집할 공시 데이터 유형"
                },
                "bsns_year": {
                    "type": "string",
                    "default": "2023",
                    "description": "사업연도 (YYYY 형식)"
                },
                "reprt_code": {
                    "type": "string",
                    "enum": ["11013", "11012", "11014", "11011"],
                    "default": "11011",
                    "description": "보고서 코드: 11011(사업보고서), 11012(반기보고서), 11013(1분기보고서), 11014(3분기보고서)"
                }
            },
            "required": ["corp_name"]
        }
    },
    {
        "name": "perform_valuation_analysis",
        "description": "수집된 데이터를 바탕으로 기업 가치평가를 수행합니다. 상대가치법(PER, PBR, EV/EBITDA)과 리스크 평가를 통합합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "yfinance_data": {
                    "type": "object",
                    "description": "Yahoo Finance에서 수집한 데이터"
                },
                "dart_data": {
                    "type": "object", 
                    "description": "DART에서 수집한 공시 데이터"
                },
                "valuation_methods": {
                    "type": "array",
                    "items": {
                        "type": "string",
                        "enum": ["per_analysis", "pbr_analysis", "ev_ebitda_analysis", "dividend_yield"]
                    },
                    "default": ["per_analysis", "pbr_analysis"],
                    "description": "사용할 가치평가 방법"
                },
                "industry_sector": {
                    "type": "string",
                    "description": "업종 분류 (동종업계 비교용, 선택사항)"
                },
                "risk_weight": {
                    "type": "number",
                    "minimum": 0,
                    "maximum": 1,
                    "default": 0.3,
                    "description": "리스크 가중치 (0: 리스크 무시, 1: 리스크 최대 반영)"
                }
            },
            "required": ["yfinance_data", "dart_data"]
        }
    },
    {
        "name": "assess_investment_risk",
        "description": "DART 공시 데이터를 기반으로 투자 리스크를 평가합니다. 재무건전성, 경영 리스크, 시장 리스크 등을 종합 분석합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "dart_data": {
                    "type": "object",
                    "description": "DART 공시 데이터"
                },
                "risk_categories": {
                    "type": "array",
                    "items": {
                        "type": "string",
                        "enum": ["financial_risk", "audit_risk", "liquidity_risk", "governance_risk", "business_risk"]
                    },
                    "default": ["financial_risk", "audit_risk", "liquidity_risk"],
                    "description": "평가할 리스크 카테고리"
                },
                "risk_threshold": {
                    "type": "string",
                    "enum": ["conservative", "moderate", "aggressive"],
                    "default": "moderate",
                    "description": "리스크 평가 기준: conservative(보수적), moderate(중간), aggressive(공격적)"
                }
            },
            "required": ["dart_data"]
        }
    },
    {
        "name": "generate_investment_recommendation",
        "description": "가치평가와 리스크 분석 결과를 종합하여 투자 판단을 생성합니다. 매수/보류/매도 권고와 근거를 제시합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "valuation_result": {
                    "type": "object",
                    "description": "가치평가 분석 결과"
                },
                "risk_assessment": {
                    "type": "object",
                    "description": "리스크 평가 결과"
                },
                "market_conditions": {
                    "type": "object",
                    "description": "시장 환경 정보 (선택사항)"
                },
                "investment_horizon": {
                    "type": "string",
                    "enum": ["short_term", "medium_term", "long_term"],
                    "default": "medium_term",
                    "description": "투자 기간: short_term(단기), medium_term(중기), long_term(장기)"
                },
                "investor_profile": {
                    "type": "string",
                    "enum": ["conservative", "moderate", "aggressive"],
                    "default": "moderate",
                    "description": "투자자 성향: conservative(보수적), moderate(중간), aggressive(공격적)"
                },
                "target_price": {
                    "type": "boolean",
                    # "default": true,
                    "description": "목표주가 산출 여부"
                }
            },
            "required": ["valuation_result", "risk_assessment"]
        }
    },
    {
        "name": "get_market_trend_data",
        "description": "거시경제 지표와 시장 심리 지표를 수집하여 전반적인 시장 환경을 분석합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "indicators": {
                    "type": "array",
                    "items": {
                        "type": "string",
                        "enum": ["vix", "interest_rate", "exchange_rate", "gdp_growth", "inflation", "sector_performance"]
                    },
                    "default": ["vix", "interest_rate", "exchange_rate"],
                    "description": "수집할 시장 지표"
                },
                "region": {
                    "type": "string",
                    "enum": ["KR", "US", "global"],
                    "default": "KR",
                    "description": "지역별 지표: KR(한국), US(미국), global(글로벌)"
                },
                "period": {
                    "type": "string",
                    "enum": ["1mo", "3mo", "6mo", "1y"],
                    "default": "3mo",
                    "description": "분석 기간"
                }
            },
            "required": ["indicators"]
        }
    },
    {
        "name": "compare_historical_performance",
        "description": "과거 매수 시점의 투자 판단을 현재 관점에서 평가합니다. 당시 지표와 현재 성과를 비교 분석합니다.",
        "parameters": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "분석 대상 종목 티커"
                },
                "purchase_date": {
                    "type": "string",
                    "format": "date",
                    "description": "매수 일자 (YYYY-MM-DD)"
                },
                "purchase_price": {
                    "type": "number",
                    "description": "매수 가격"
                },
                "current_date": {
                    "type": "string",
                    "format": "date",
                    "description": "평가 기준일 (기본값: 오늘)"
                },
                "analysis_depth": {
                    "type": "string",
                    "enum": ["basic", "detailed", "comprehensive"],
                    "default": "detailed",
                    "description": "분석 깊이: basic(기본), detailed(상세), comprehensive(종합)"
                }
            },
            "required": ["ticker", "purchase_date", "purchase_price"]
        }
    }
]

# 실제 함수 구현들

def extract_stock_info(user_query: str, stock_name: str, analysis_intent: str, 
                      purchase_price: Optional[float] = None, 
                      purchase_date: Optional[str] = None) -> Dict[str, Any]:
    """
    사용자 질문에서 종목 정보를 추출하고 티커로 매핑
    """
    try:
        # 종목명 → 티커 매핑 로직
        ticker = map_stock_name_to_ticker(stock_name)
        
        if not ticker:
            return {
                "status": "error",
                "message": f"'{stock_name}' 종목을 찾을 수 없습니다. 정확한 종목명을 입력해주세요.",
                # "suggestions": suggest_similar_stocks(stock_name)
            }
        
        # 상장 여부 확인
        is_listed = verify_stock_listing(ticker)
        
        result = {
            "status": "success",
            "original_query": user_query,
            "stock_name": stock_name,
            "ticker": ticker,
            "market": get_market_type(ticker),  # KR, US, etc.
            "is_listed": is_listed,
            "analysis_intent": analysis_intent,
            "purchase_info": {
                "price": purchase_price,
                "date": purchase_date
            } if purchase_price and purchase_date else None
        }
        
        return result
        
    except Exception as e:
        return {
            "status": "error",
            "message": f"종목 정보 추출 중 오류 발생: {str(e)}"
        }


In [4]:

def get_yfinance_data(ticker: str, data_types: List[str] = None, period: str = "1y") -> Dict[str, Any]:
    """
    Yahoo Finance에서 종합 데이터 수집
    """
    if data_types is None:
        data_types = ["price_data", "financial_statements", "analyst_info", "news"]
    
    try:
        stock = yf.Ticker(ticker)
        result = {"ticker": ticker, "data": {}}
        
        # 기본 정보
        result["data"]["info"] = stock.info
        
        # 가격 데이터
        if "price_data" in data_types:
            result["data"]["price_data"] = {
                "current_price": stock.info.get("currentPrice"),
                "previous_close": stock.info.get("previousClose"),
                "day_high": stock.info.get("dayHigh"),
                "day_low": stock.info.get("dayLow"),
                "52_week_high": stock.info.get("fiftyTwoWeekHigh"),
                "52_week_low": stock.info.get("fiftyTwoWeekLow"),
                "market_cap": stock.info.get("marketCap"),
                "volume": stock.info.get("volume"),
                "avg_volume": stock.info.get("averageVolume")
            }
            
            # 주가 히스토리
            hist = stock.history(period=period)
            result["data"]["price_history"] = hist.to_dict()
        
        # 재무제표
        if "financial_statements" in data_types:
            result["data"]["financials"] = {
                "income_statement": stock.financials.to_dict() if not stock.financials.empty else {},
                "balance_sheet": stock.balance_sheet.to_dict() if not stock.balance_sheet.empty else {},
                "cash_flow": stock.cashflow.to_dict() if not stock.cashflow.empty else {}
            }
        
        # 애널리스트 정보
        if "analyst_info" in data_types:
            result["data"]["analyst_info"] = {
                "recommendations": stock.recommendations.to_dict() if stock.recommendations is not None else {},
                "target_price": stock.info.get("targetMeanPrice"),
                "recommendation_key": stock.info.get("recommendationKey"),
                "number_of_analysts": stock.info.get("numberOfAnalystOpinions")
            }
        
        # 뉴스
        if "news" in data_types:
            news = stock.news[:5]  # 최근 5개 뉴스만
            result["data"]["news"] = [
                {
                    "title": item.get("title"),
                    "publisher": item.get("publisher"),
                    "published_at": datetime.fromtimestamp(item.get("providerPublishTime", 0)).strftime("%Y-%m-%d %H:%M"),
                    "url": item.get("link")
                } for item in news
            ]
        
        # 기술적 지표
        if "technical_indicators" in data_types:
            hist = stock.history(period="3mo")
            if not hist.empty:
                result["data"]["technical_indicators"] = calculate_technical_indicators(hist)
        
        return {"status": "success", "data": result}
        
    except Exception as e:
        return {
            "status": "error", 
            "message": f"Yahoo Finance 데이터 수집 오류: {str(e)}"
        }

def get_dart_disclosure_data(corp_name: str, corp_code: str = None, 
                           data_types: List[str] = None, bsns_year: str = "2023",
                           reprt_code: str = "11011") -> Dict[str, Any]:
    """
    DART 전자공시 데이터 수집
    """
    if data_types is None:
        data_types = ["company_info", "financial_summary", "audit_report"]
    
    try:
        dart_api_key = get_dart_api_key()  # 환경변수에서 API 키 로드
        result = {"corp_name": corp_name, "data": {}}
        
        # 기업 코드가 없으면 검색
        if not corp_code:
            corp_code = search_dart_corp_code(corp_name, dart_api_key)
            if not corp_code:
                return {"status": "error", "message": f"'{corp_name}' 기업 코드를 찾을 수 없습니다."}
        
        # 기업 개요
        if "company_info" in data_types:
            company_info = get_dart_company_info(corp_code, dart_api_key)
            result["data"]["company_info"] = company_info
        
        # 재무제표 요약
        if "financial_summary" in data_types:
            financial_data = get_dart_financial_summary(corp_code, bsns_year, reprt_code, dart_api_key)
            result["data"]["financial_summary"] = financial_data
        
        # 다년도 재무 비교
        if "multi_year_financial" in data_types:
            multi_year_data = get_dart_multi_year_financial(corp_code, dart_api_key)
            result["data"]["multi_year_financial"] = multi_year_data
        
        # 감사보고서 분석
        if "audit_report" in data_types:
            audit_data = get_dart_audit_report(corp_code, bsns_year, reprt_code, dart_api_key)
            result["data"]["audit_report"] = audit_data
        
        # 주요 공시사항
        if "major_disclosure" in data_types:
            disclosure_data = get_dart_major_disclosures(corp_code, dart_api_key)
            result["data"]["major_disclosure"] = disclosure_data
        
        return {"status": "success", "data": result}
        
    except Exception as e:
        return {
            "status": "error",
            "message": f"DART 데이터 수집 오류: {str(e)}"
        }

def perform_valuation_analysis(yfinance_data: Dict, dart_data: Dict,
                             valuation_methods: List[str] = None,
                             industry_sector: str = None,
                             risk_weight: float = 0.3) -> Dict[str, Any]:
    """
    종합 가치평가 분석 수행
    """
    if valuation_methods is None:
        valuation_methods = ["per_analysis", "pbr_analysis"]
    
    try:
        result = {
            "valuation_summary": {},
            "detailed_analysis": {},
            "recommendation": ""
        }
        
        # 현재 주가 및 기본 지표
        current_price = yfinance_data["data"]["price_data"]["current_price"]
        market_cap = yfinance_data["data"]["price_data"]["market_cap"]
        
        # PER 분석
        if "per_analysis" in valuation_methods:
            per_analysis = calculate_per_analysis(yfinance_data, dart_data, industry_sector)
            result["detailed_analysis"]["per_analysis"] = per_analysis
        
        # PBR 분석  
        if "pbr_analysis" in valuation_methods:
            pbr_analysis = calculate_pbr_analysis(yfinance_data, dart_data, industry_sector)
            result["detailed_analysis"]["pbr_analysis"] = pbr_analysis
        
        # EV/EBITDA 분석
        if "ev_ebitda_analysis" in valuation_methods:
            ev_ebitda_analysis = calculate_ev_ebitda_analysis(yfinance_data, dart_data, industry_sector)
            result["detailed_analysis"]["ev_ebitda_analysis"] = ev_ebitda_analysis
        
        # 배당수익률 분석
        if "dividend_yield" in valuation_methods:
            dividend_analysis = calculate_dividend_analysis(yfinance_data, dart_data)
            result["detailed_analysis"]["dividend_analysis"] = dividend_analysis
        
        # 종합 평가 점수 계산
        valuation_score = calculate_composite_valuation_score(result["detailed_analysis"], risk_weight)
        result["valuation_summary"] = valuation_score
        
        return {"status": "success", "data": result}
        
    except Exception as e:
        return {
            "status": "error",
            "message": f"가치평가 분석 오류: {str(e)}"
        }

def assess_investment_risk(dart_data: Dict, risk_categories: List[str] = None,
                         risk_threshold: str = "moderate") -> Dict[str, Any]:
    """
    투자 리스크 종합 평가
    """
    if risk_categories is None:
        risk_categories = ["financial_risk", "audit_risk", "liquidity_risk"]
    
    try:
        result = {
            "overall_risk_score": 0,
            "risk_level": "low",  # low, moderate, high
            "risk_factors": {},
            "warnings": []
        }
        
        risk_scores = []
        
        # 재무 리스크
        if "financial_risk" in risk_categories:
            financial_risk = assess_financial_risk(dart_data, risk_threshold)
            result["risk_factors"]["financial_risk"] = financial_risk
            risk_scores.append(financial_risk["score"])
        
        # 감사 리스크
        if "audit_risk" in risk_categories:
            audit_risk = assess_audit_risk(dart_data, risk_threshold)
            result["risk_factors"]["audit_risk"] = audit_risk
            risk_scores.append(audit_risk["score"])
        
        # 유동성 리스크
        if "liquidity_risk" in risk_categories:
            liquidity_risk = assess_liquidity_risk(dart_data, risk_threshold)
            result["risk_factors"]["liquidity_risk"] = liquidity_risk
            risk_scores.append(liquidity_risk["score"])
        
        # 지배구조 리스크
        if "governance_risk" in risk_categories:
            governance_risk = assess_governance_risk(dart_data, risk_threshold)
            result["risk_factors"]["governance_risk"] = governance_risk
            risk_scores.append(governance_risk["score"])
        
        # 사업 리스크
        if "business_risk" in risk_categories:
            business_risk = assess_business_risk(dart_data, risk_threshold)
            result["risk_factors"]["business_risk"] = business_risk
            risk_scores.append(business_risk["score"])
        
        # 종합 리스크 점수 계산
        if risk_scores:
            result["overall_risk_score"] = sum(risk_scores) / len(risk_scores)
            
            if result["overall_risk_score"] < 3:
                result["risk_level"] = "low"
            elif result["overall_risk_score"] < 7:
                result["risk_level"] = "moderate" 
            else:
                result["risk_level"] = "high"
        
        # 주요 경고사항 추출
        result["warnings"] = extract_risk_warnings(result["risk_factors"])
        
        return {"status": "success", "data": result}
        
    except Exception as e:
        return {
            "status": "error",
            "message": f"리스크 평가 오류: {str(e)}"
        }

def generate_investment_recommendation(valuation_result: Dict, risk_assessment: Dict,
                                     market_conditions: Dict = None,
                                     investment_horizon: str = "medium_term",
                                     investor_profile: str = "moderate",
                                     target_price: bool = True) -> Dict[str, Any]:
    """
    최종 투자 판단 및 권고 생성
    """
    try:
        result = {
            "recommendation": "",  # BUY, HOLD, SELL
            "confidence_level": 0,  # 0-100
            "target_price_range": {},
            "reasoning": [],
            "action_plan": [],
            "risk_warnings": []
        }
        
        # 가치평가 점수와 리스크 점수 추출
        valuation_score = valuation_result.get("valuation_summary", {}).get("overall_score", 5)
        risk_score = risk_assessment.get("overall_risk_score", 5)
        risk_level = risk_assessment.get("risk_level", "moderate")
        
        # 매트릭스 기반 투자 판단
        recommendation_matrix = {
            ("low", "low"): ("BUY", 85),      # 저평가 + 저리스크
            ("low", "moderate"): ("BUY", 70),  # 저평가 + 중리스크  
            ("low", "high"): ("HOLD", 55),     # 저평가 + 고리스크
            ("fair", "low"): ("HOLD", 65),     # 적정가 + 저리스크
            ("fair", "moderate"): ("HOLD", 50), # 적정가 + 중리스크
            ("fair", "high"): ("SELL", 45),    # 적정가 + 고리스크
            ("high", "low"): ("HOLD", 40),     # 고평가 + 저리스크
            ("high", "moderate"): ("SELL", 65), # 고평가 + 중리스크
            ("high", "high"): ("SELL", 80)     # 고평가 + 고리스크
        }
        
        # 가치평가 레벨 결정
        if valuation_score < 4:
            valuation_level = "low"  # 저평가
        elif valuation_score < 7:
            valuation_level = "fair"  # 적정가
        else:
            valuation_level = "high"  # 고평가
        
        # 기본 추천 결정
        base_recommendation, base_confidence = recommendation_matrix.get(
            (valuation_level, risk_level), ("HOLD", 50)
        )
        
        # 투자자 성향 및 투자 기간 반영
        adjusted_recommendation, adjusted_confidence = adjust_recommendation_by_profile(
            base_recommendation, base_confidence, investor_profile, investment_horizon
        )
        
        result["recommendation"] = adjusted_recommendation
        result["confidence_level"] = adjusted_confidence
        
        # 목표가 산출
        if target_price:
            result["target_price_range"] = calculate_target_price_range(valuation_result)
        
        # 판단 근거 생성
        result["reasoning"] = generate_investment_reasoning(
            valuation_result, risk_assessment, adjusted_recommendation
        )
        
        # 액션 플랜 생성
        result["action_plan"] = generate_action_plan(
            adjusted_recommendation, investment_horizon, investor_profile
        )
        
        # 리스크 경고사항
        result["risk_warnings"] = risk_assessment.get("warnings", [])
        
        return {"status": "success", "data": result}
        
    except Exception as e:
        return {
            "status": "error",
            "message": f"투자 판단 생성 오류: {str(e)}"
        }

def get_market_trend_data(indicators: List[str], region: str = "KR", period: str = "3mo") -> Dict[str, Any]:
    """
    거시경제 지표 및 시장 심리 지표 수집
    """
    try:
        result = {"region": region, "period": period, "indicators": {}}
        
        # VIX 지수 (공포지수)
        if "vix" in indicators:
            vix_data = get_vix_data(period)
            result["indicators"]["vix"] = vix_data
        
        # 금리 데이터
        if "interest_rate" in indicators:
            if region == "KR":
                rate_data = get_korea_interest_rate(period)
            elif region == "US":
                rate_data = get_us_interest_rate(period)
            else:
                rate_data = get_global_interest_rate(period)
            result["indicators"]["interest_rate"] = rate_data
        
        # 환율 데이터
        if "exchange_rate" in indicators:
            exchange_data = get_exchange_rate_data(region, period)
            result["indicators"]["exchange_rate"] = exchange_data
        
        # GDP 성장률
        if "gdp_growth" in indicators:
            gdp_data = get_gdp_growth_data(region, period)
            result["indicators"]["gdp_growth"] = gdp_data
        
        # 인플레이션
        if "inflation" in indicators:
            inflation_data = get_inflation_data(region, period)
            result["indicators"]["inflation"] = inflation_data
        
        # 섹터 성과
        if "sector_performance" in indicators:
            sector_data = get_sector_performance_data(region, period)
            result["indicators"]["sector_performance"] = sector_data
        
        return {"status": "success", "data": result}
        
    except Exception as e:
        return {
            "status": "error",
            "message": f"시장 데이터 수집 오류: {str(e)}"
        }

def compare_historical_performance(ticker: str, purchase_date: str, purchase_price: float,
                                 current_date: str = None, analysis_depth: str = "detailed") -> Dict[str, Any]:
    """
    과거 매수 시점 대비 현재 성과 분석
    """
    try:
        if not current_date:
            current_date = datetime.now().strftime("%Y-%m-%d")
        
        result = {
            "purchase_info": {
                "date": purchase_date,
                "price": purchase_price
            },
            "performance_analysis": {},
            "contextual_analysis": {},
            "lessons_learned": []
        }
        
        # 현재 주가 정보
        stock = yf.Ticker(ticker)
        current_info = stock.info
        current_price = current_info.get("currentPrice", 0)
        
        # 수익률 계산
        total_return = ((current_price - purchase_price) / purchase_price) * 100
        days_held = (datetime.strptime(current_date, "%Y-%m-%d") - 
                    datetime.strptime(purchase_date, "%Y-%m-%d")).days
        annualized_return = (((current_price / purchase_price) ** (365 / days_held)) - 1) * 100
        
        result["performance_analysis"] = {
            "current_price": current_price,
            "total_return_pct": total_return,
            "total_return_amount": current_price - purchase_price,
            "days_held": days_held,
            "annualized_return_pct": annualized_return
        }
        
        # 상세 분석
        if analysis_depth in ["detailed", "comprehensive"]:
            # 매수 시점 컨텍스트 분석
            purchase_context = analyze_purchase_context(ticker, purchase_date, purchase_price)
            result["contextual_analysis"]["purchase_context"] = purchase_context
            
            # 시장 대비 성과
            market_comparison = compare_against_market(ticker, purchase_date, current_date)
            result["contextual_analysis"]["market_comparison"] = market_comparison
        
        # 종합 분석
        if analysis_depth == "comprehensive":
            # 투자 교훈 추출
            lessons = extract_investment_lessons(result, ticker)
            result["lessons_learned"] = lessons
            
            # 당시 vs 현재 밸류에이션 비교
            valuation_comparison = compare_valuation_metrics(ticker, purchase_date, current_date)
            result["contextual_analysis"]["valuation_comparison"] = valuation_comparison
        
        return {"status": "success", "data": result}
        
    except Exception as e:
        return {
            "status": "error",
            "message": f"과거 성과 분석 오류: {str(e)}"
        }

# 보조 함수들 구현

def map_stock_name_to_ticker(stock_name: str) -> str:
    """종목명을 티커로 매핑"""
    # 한국 주식 매핑 테이블
    kr_stock_mapping = {
        "삼성전자": "005930.KS",
        "SK하이닉스": "000660.KS", 
        "NAVER": "035420.KS",
        "카카오": "035720.KS",
        "LG에너지솔루션": "373220.KS",
        "한화솔루션": "009830.KS",
        # ... 더 많은 매핑 추가
    }
    
    # 미국 주식은 그대로 사용하거나 .US 추가
    us_stocks = ["AAPL", "GOOGL", "MSFT", "TSLA", "NVDA", "AMZN"]
    
    if stock_name in kr_stock_mapping:
        return kr_stock_mapping[stock_name]
    elif stock_name.upper() in us_stocks:
        return stock_name.upper()
    elif stock_name.upper() == "APPLE":
        return "AAPL"
    elif stock_name.upper() == "GOOGLE":
        return "GOOGL"
    
    return None

def get_market_type(ticker: str) -> str:
    """티커에서 시장 유형 추출"""
    if ".KS" in ticker or ".KQ" in ticker:
        return "KR"
    elif ".US" in ticker or ticker.upper() in ["AAPL", "GOOGL", "MSFT", "TSLA", "NVDA"]:
        return "US"
    return "OTHER"

def verify_stock_listing(ticker: str) -> bool:
    """상장 여부 확인"""
    try:
        stock = yf.Ticker(ticker)
        info = stock.info
        return bool(info.get("symbol"))
    except:
        return False

def calculate_technical_indicators(hist_data: pd.DataFrame) -> Dict:
    """기술적 지표 계산"""
    try:
        result = {}
        
        # RSI 계산
        delta = hist_data['Close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        rs = gain / loss
        result['rsi'] = (100 - (100 / (1 + rs))).iloc[-1]
        
        # 이동평균
        result['ma_20'] = hist_data['Close'].rolling(20).mean().iloc[-1]
        result['ma_50'] = hist_data['Close'].rolling(50).mean().iloc[-1]
        
        # 볼린저 밴드
        bb_period = 20
        bb_std = 2
        bb_ma = hist_data['Close'].rolling(bb_period).mean()
        bb_std_dev = hist_data['Close'].rolling(bb_period).std()
        result['bollinger_upper'] = (bb_ma + bb_std_dev * bb_std).iloc[-1]
        result['bollinger_lower'] = (bb_ma - bb_std_dev * bb_std).iloc[-1]
        
        return result
    except:
        return {}

def get_dart_api_key() -> str:
    """환경변수에서 DART API 키 로드"""
    import os
    return os.getenv("DART_API_KEY", "your_dart_api_key_here")

def search_dart_corp_code(corp_name: str, api_key: str) -> str:
    """기업명으로 DART 기업코드 검색"""
    try:
        url = "https://opendart.fss.or.kr/api/company.json"
        params = {
            "crtfc_key": api_key,
            "corp_name": corp_name
        }
        response = requests.get(url, params=params)
        data = response.json()
        
        if data.get("status") == "000":
            return data["list"][0]["corp_code"] if data.get("list") else None
        return None
    except:
        return None

def calculate_per_analysis(yfinance_data: Dict, dart_data: Dict, industry_sector: str = None) -> Dict:
    """PER 분석"""
    try:
        info = yfinance_data["data"]["info"]
        current_pe = info.get("trailingPE")
        forward_pe = info.get("forwardPE")
        
        # 업종 평균 PER (임시 데이터)
        industry_avg_pe = get_industry_average_pe(industry_sector) if industry_sector else 15.0
        
        analysis = {
            "current_pe": current_pe,
            "forward_pe": forward_pe,
            "industry_avg_pe": industry_avg_pe,
            "pe_premium_discount": ((current_pe / industry_avg_pe - 1) * 100) if current_pe and industry_avg_pe else None,
            "valuation_signal": "undervalued" if current_pe and current_pe < industry_avg_pe else "overvalued"
        }
        
        return analysis
    except:
        return {}

def calculate_composite_valuation_score(detailed_analysis: Dict, risk_weight: float) -> Dict:
    """종합 밸류에이션 점수 계산"""
    try:
        scores = []
        
        # PER 점수
        per_analysis = detailed_analysis.get("per_analysis", {})
        if per_analysis.get("current_pe"):
            pe_score = calculate_pe_score(per_analysis)
            scores.append(pe_score)
        
        # PBR 점수
        pbr_analysis = detailed_analysis.get("pbr_analysis", {})
        if pbr_analysis:
            pbr_score = calculate_pbr_score(pbr_analysis)
            scores.append(pbr_score)
        
        # 평균 점수 계산
        overall_score = sum(scores) / len(scores) if scores else 5.0
        
        # 리스크 가중치 적용
        risk_adjusted_score = overall_score * (1 - risk_weight) + 5 * risk_weight
        
        return {
            "overall_score": risk_adjusted_score,
            "component_scores": {
                "per_score": scores[0] if len(scores) > 0 else None,
                "pbr_score": scores[1] if len(scores) > 1 else None
            },
            "valuation_level": "undervalued" if risk_adjusted_score < 4 else "overvalued" if risk_adjusted_score > 6 else "fair"
        }
    except:
        return {"overall_score": 5.0, "valuation_level": "fair"}

def assess_financial_risk(dart_data: Dict, threshold: str) -> Dict:
    """재무 리스크 평가"""
    try:
        financial_data = dart_data.get("data", {}).get("financial_summary", {})
        
        # 부채비율, 유동비율 등 계산
        debt_ratio = calculate_debt_ratio(financial_data)
        current_ratio = calculate_current_ratio(financial_data)
        
        risk_score = 0
        risk_factors = []
        
        # 부채비율 체크
        if debt_ratio > 200:  # 200% 초과시 고위험
            risk_score += 3
            risk_factors.append("높은 부채비율")
        
        # 유동비율 체크  
        if current_ratio < 100:  # 100% 미만시 위험
            risk_score += 2
            risk_factors.append("낮은 유동비율")
        
        return {
            "score": min(risk_score, 10),
            "factors": risk_factors,
            "metrics": {
                "debt_ratio": debt_ratio,
                "current_ratio": current_ratio
            }
        }
    except:
        return {"score": 5, "factors": [], "metrics": {}}

# # 추가 보조 함수들은 실제 구현에서 완성 필요
# def suggest_similar_stocks(stock_name: str) -> List[str]:
#     """유사 종목명 추천"""
#     return []

# def get_dart_company_info(corp_code: str, api_key: str) -> Dict:
#     """DART 기업 개요 조회"""
#     return {}

# def get_dart_financial_summary(corp_code: str, bsns_year: str, reprt_code: str, api_key: str) -> Dict:
#     """DART 재무제표 요약 조회"""
#     return {}

# def get_dart_audit_report(corp_code: str, bsns_year: str, reprt_code: str, api_key: str) -> Dict:
#     """DART 감사보고서 조회"""
#     return {}

# def calculate_target_price_range(valuation_result: Dict) -> Dict:
#     """목표가 범위 계산"""
#     return {"low": 0, "mid": 0, "high": 0}

# def generate_investment_reasoning(valuation_result: Dict, risk_assessment: Dict, recommendation: str) -> List[str]:
#     """투자 판단 근거 생성"""
#     return []


In [ ]:
import yfinance as yf
from datetime import datetime
from typing import List, Dict, Any


# 테스트 실행
if __name__ == "__main__":
    ticker = "AAPL"  # 애플 주식
    data_types = ["price_data", "financial_statements", "analyst_info", "news"]
    period = "6mo"

    result = get_yfinance_data(ticker, data_types, period)

    if result["status"] == "success":
        pprint(result)

{'status': 'success', 'data': {'ticker': 'AAPL', 'data': {'info': {'address1': 'One Apple Park Way', 'city': 'Cupertino', 'state': 'CA', 'zip': '95014', 'country': 'United States', 'phone': '(408) 996-1010', 'website': 'https://www.apple.com', 'industry': 'Consumer Electronics', 'industryKey': 'consumer-electronics', 'industryDisp': 'Consumer Electronics', 'sector': 'Technology', 'sectorKey': 'technology', 'sectorDisp': 'Technology', 'longBusinessSummary': 'Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple TV, Apple Watch, Beats products, and HomePod. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allow customers to discover and download applications and digit

In [3]:

# 주요 사용 예시
if __name__ == "__main__":
    # 예시 1: 종목 정보 추출
    stock_info = extract_stock_info(
        user_query="삼성전자 지금 사도 될까요?",
        stock_name="삼성전자", 
        analysis_intent="buy_decision"
    )
    print("종목 정보:", stock_info)
    
    # 예시 2: 과거 매수 평가
    historical_analysis = compare_historical_performance(
        ticker="005930.KS",
        purchase_date="2023-07-15", 
        purchase_price=70000,
        analysis_depth="comprehensive"
    )
    print("과거 매수 분석:", historical_analysis)

종목 정보: {'status': 'success', 'original_query': '삼성전자 지금 사도 될까요?', 'stock_name': '삼성전자', 'ticker': '005930.KS', 'market': 'KR', 'is_listed': True, 'analysis_intent': 'buy_decision', 'purchase_info': None}
과거 매수 분석: {'status': 'error', 'message': "과거 성과 분석 오류: name 'analyze_purchase_context' is not defined"}


In [ ]:
핵심 기능 구조
1. 종목 정보 추출 및 매핑

extract_stock_info(): 사용자 질문에서 종목명 추출하고 티커로 변환
한국/미국 시장 모두 지원, 상장 여부 검증

2. 데이터 수집 모듈

get_yfinance_data(): Yahoo Finance에서 성장성 지표 수집 (시세, 재무제표, 애널리스트 의견, 뉴스)
get_dart_disclosure_data(): DART에서 실적/리스크 지표 수집 (공시, 감사보고서, 재무제표)
get_market_trend_data(): 거시경제 지표 수집 (VIX, 금리, 환율, GDP 등)

3. 분석 엔진

perform_valuation_analysis(): 상대가치법 기반 밸류에이션 (PER, PBR, EV/EBITDA)
assess_investment_risk(): 다차원 리스크 평가 (재무, 감사, 유동성, 지배구조 리스크)

4. 투자 판단 생성

generate_investment_recommendation(): 매수/보류/매도 권고 및 목표가 산출
compare_historical_performance(): 과거 매수 시점 대비 성과 분석

주요 특징
✅ 통합 데이터 소스: yfinance + DART API 결합으로 성장성과 리스크 동시 평가
✅ 다차원 분석: 정량적 밸류에이션 + 정성적 리스크 평가
✅ 개인화 추천: 투자자 성향과 투자 기간 반영
✅ 멀티턴 대화: 과거 매수 평가, 현재 매수 판단, 종목 탐색 등 다양한 시나리오 지원